[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/02_linear_regression/02_linear_regression.ipynb)

# 02. Linear Regression

> 원본 강의: [Lec 1–4, 모두를 위한 머신러닝과 딥러닝](https://hunkim.github.io/ml/) — ML 개념/용어, Linear Regression, Cost 함수 최소화, 다중 입력 Linear Regression

## 이론

### 1) 머신러닝이란
명시적인 규칙을 사람이 하나하나 코딩하는 대신, **데이터로부터 패턴을 학습**시켜 프로그램을 만드는 방법입니다.
- **지도학습(Supervised)**: 정답(label)이 있는 데이터로 학습 — 회귀(연속값 예측), 분류(범주 예측)
- **비지도학습(Unsupervised)**: 정답 없이 데이터의 구조(군집 등)를 학습

### 2) Linear Regression의 가설(Hypothesis)
$$H(x) = Wx + b$$
$W$(가중치)와 $b$(편향)를 데이터에 맞게 조정해서, 입력 $x$로부터 출력 $y$를 잘 예측하는 직선을 찾는 것이 목표입니다.

### 3) Cost(=Loss) 함수
예측이 실제값과 얼마나 다른지를 수치화합니다. 평균 제곱 오차(MSE)를 사용합니다.
$$J(W, b) = \frac{1}{m}\sum_{i=1}^{m}\left(H(x^{(i)}) - y^{(i)}\right)^2$$
$J$가 작을수록 좋은 모델입니다. 학습 = $J(W,b)$를 최소화하는 $W, b$를 찾는 과정.

### 4) Gradient Descent (경사 하강법)
$J$를 $W$에 대해 미분한 기울기 방향의 반대로 조금씩 $W$를 이동시킵니다.
$$W := W - \alpha \frac{\partial}{\partial W}J(W, b)$$
$\alpha$는 **학습률(learning rate)**로, 너무 크면 발산하고 너무 작으면 학습이 느립니다.

### 5) 다중 변수(Multi-variable) Linear Regression
입력이 여러 개인 경우 행렬로 표현합니다.
$$H(X) = XW + b$$
여기서 $X$는 $(m \times n)$ 행렬($m$=샘플 수, $n$=특성 수), $W$는 $(n \times 1)$ 벡터입니다.

## 실습 0. Colab 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)
if IN_COLAB:
    !pip install -q scikit-learn numpy matplotlib

## 실습 1. Gradient Descent를 numpy로 직접 구현 (단순 선형회귀)

라이브러리 없이 경사 하강법을 직접 구현해보면서, Cost가 줄어드는 과정을 눈으로 확인합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 공부 시간(x) -> 시험 점수(y), y = 2x + 3 근처의 노이즈 섞인 데이터
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 50)
y = 2 * x + 3 + rng.normal(0, 1.5, size=x.shape)

W, b = 0.0, 0.0
lr = 0.01
epochs = 200
cost_history = []

m = len(x)
for epoch in range(epochs):
    y_pred = W * x + b
    cost = np.mean((y_pred - y) ** 2)
    cost_history.append(cost)

    # dJ/dW, dJ/db
    dW = (2 / m) * np.sum((y_pred - y) * x)
    db = (2 / m) * np.sum(y_pred - y)

    W -= lr * dW
    b -= lr * db

    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}  cost={cost:.4f}  W={W:.3f}  b={b:.3f}")

print(f"\n최종: W={W:.3f}, b={b:.3f} (목표: W=2, b=3)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(cost_history)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cost (MSE)")
axes[0].set_title("Cost가 줄어드는 과정")

axes[1].scatter(x, y, label="data")
axes[1].plot(x, W * x + b, color="red", label="학습된 H(x)")
axes[1].set_title("학습 결과")
axes[1].legend()
plt.show()

## 실습 2. 다중 변수 Linear Regression — 파일에서 데이터 로딩

원본 강의의 Lab("파일 데이터 로딩")처럼, CSV 파일을 만들어 `np.loadtxt`로 읽고 scikit-learn으로 학습합니다.

In [ ]:
import os

os.makedirs("../../data", exist_ok=True)

# 퀴즈1, 퀴즈2, 중간고사 점수 -> 기말고사 점수 (다중 입력 회귀)
rng = np.random.default_rng(0)
n_samples = 25
quiz1 = rng.integers(60, 100, n_samples)
quiz2 = rng.integers(60, 100, n_samples)
midterm = rng.integers(60, 100, n_samples)
final = (0.3 * quiz1 + 0.3 * quiz2 + 0.4 * midterm + rng.normal(0, 3, n_samples)).round(1)

data = np.column_stack([quiz1, quiz2, midterm, final])
np.savetxt("../../data/scores.csv", data, delimiter=",", fmt="%.1f",
           header="quiz1,quiz2,midterm,final", comments="")
print("data/scores.csv 생성 완료")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

loaded = np.loadtxt("../../data/scores.csv", delimiter=",", skiprows=1)
X = loaded[:, :3]   # quiz1, quiz2, midterm
y = loaded[:, 3]    # final

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

reg = LinearRegression()
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

print("W (계수):", reg.coef_)
print("b (절편):", reg.intercept_)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R^2:", r2_score(y_test, y_pred))

## 정리 & 연습 문제
- Cost 함수가 매 epoch 감소하는 걸 확인했다면, `lr`(학습률)을 0.1이나 0.001로 바꿔보고 무슨 일이 일어나는지 관찰해보세요.
- 실습 2에서 특성을 하나 더 추가(예: 출석률)해서 성능이 좋아지는지 확인해보세요.
- 다음 노트북([03_classification.ipynb](../03_classification/03_classification.ipynb))에서는 연속값이 아닌 "범주"를 예측하는 분류 문제로 넘어갑니다.